# Chung–Lu degree-corrected colocalization — doublets + triplets (real PNA)

Per-cell, degree-corrected colocalization under the **Chung–Lu null** (analytic, no MCMC):
**doublets** (marker-pair join counts `J_AB`) and **triplets** (unordered open-wedge counts `W_{ABC}`).

Each PNA cell is a molecule-level graph: node = one UMI carrying one marker, edge = proximity ligation.
Under the soft Chung–Lu null (`p_ij = d_i d_j / 2E`) every motif count has a closed-form mean/variance
in the per-marker degree-moment strengths `s,r,t,u`. Math + driver live in `chunglu_triplets.py`;
the full per-sample run is on LSF (`run_chunglu_triplets.lsf`). See the method spec in
`PROMPT_run_chunglu_on_real_pna.md`.

**Flow:** Step 0 inspect → Step 1 MCMC validation gate → Step 2 submit LSF jobs → Step 3 aggregate.

In [ ]:
import sys, json, time, random
sys.path.insert(0, "/home/projects/nyosef/zvise/PixelGen/PixelGen")
sys.path.insert(0, "/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data")
import numpy as np, pandas as pd, scanpy as sc
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from itertools import combinations
from pixelator import read_pna
import chunglu_triplets as ct

sns.set_style("whitegrid")
plt.rcParams.update({"figure.figsize": (5, 3), "axes.titlesize": 11, "font.family": "sans-serif"})

BASE = Path("/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data")
RESULTS = BASE / "results"
CACHE = BASE / "cache"
OUT = RESULTS / "chunglu"
OUT.mkdir(exist_ok=True)
ADATA_PATH = CACHE / "adata_annotated.h5ad"
SAMPLES = ["S001", "S002", "S003", "S004", "S005", "S006", "S007", "S008",
           "S009", "S010", "S011", "S012", "S013", "S014", "S016"]
def pxl_path(s): return RESULTS / s / "layout" / "layout" / f"{s}.layout.pxl"

adata = sc.read_h5ad(ADATA_PATH, backed="r")
print("adata:", adata.shape)
print("cells per sample:", adata.obs["sample"].value_counts().sort_index().to_dict())

In [ ]:
# --- Per-cell triplet wedge (closed candidate list) — submit + track. Re-run to refresh. ---
# Builds triplet_candidate_list.json if missing, submits run_chunglu_triplets_percell.lsf,
# and will NOT resubmit while clpc jobs are queued or all 15 outputs already exist.
(RESULTS / "logs").mkdir(parents=True, exist_ok=True)
LIST_JSON = OUT / "triplet_candidate_list.json"
if not LIST_JSON.exists():
    print("building closed list ->", LIST_JSON.name)
    !cd {BASE} && {sys.executable} build_triplet_list.py
n_trip = len(json.loads(LIST_JSON.read_text())["triples"])

_jobs = !bjobs -w 2>/dev/null
qjobs = [l for l in _jobs if "clpc" in l.lower()]
done = sorted(p.name.split("_")[0] for p in OUT.glob("*_triplets_percell.parquet"))

if qjobs:
    print(f"clpc jobs already in queue ({len(qjobs)}):"); print("\n".join(qjobs))
elif len(done) >= len(SAMPLES):
    print(f"all {len(done)}/{len(SAMPLES)} per-cell outputs present — nothing to submit.")
else:
    print(f"submitting per-cell array: {n_trip} triplets × {len(SAMPLES)} samples ...")
    !cd {BASE} && bsub < run_chunglu_triplets_percell.lsf

print(f"\n{len(done)}/{len(SAMPLES)} samples done:", done)
miss = [s for s in SAMPLES if s not in done]
print("waiting on:", miss if miss else "none — complete")
!tail -n 3 {RESULTS}/logs/clpc_*.err 2>/dev/null | tail -n 20

## Step 0 — Inspect data & canonical form

In [ ]:
# Global marker list (stable K across all samples) + save for the LSF driver.
marker_names = adata.var_names.tolist()
K = len(marker_names)
marker_to_idx = {m: i for i, m in enumerate(marker_names)}
(OUT / "marker_names.json").write_text(json.dumps(marker_names))
print("K =", K)

# Inspect one cell's edgelist + canonical form. The PNA graph is bipartite (umi1/umi2
# sides), so the null is computed per side (normaliser m = edge count, not 2E).
pg = read_pna([pxl_path("S001")])
comp = list(pg.edgelist().components)[0]
el = pg.filter(components=[comp]).edgelist().to_df()
print("edgelist cols:", el.columns.tolist(), "shape:", el.shape)
print("umi1/umi2 overlap:", len(set(el["umi1"]) & set(el["umi2"])), "(bipartite -> 0)")

edges, labels, side, n = ct.load_cell(el[["umi1", "umi2", "marker_1", "marker_2"]], marker_to_idx)
A, L, d, twoE, N = ct.build_cell(edges, labels, K)
m = d.sum() / 2
print(f"n={n} (n1={int((side==1).sum())} n2={int((side==2).sum())})  "
      f"E={edges.shape[0]}  m={m:.0f}  2E={twoE:.0f}  "
      f"deg mean/med/max={d.mean():.1f}/{np.median(d):.0f}/{d.max():.0f}")
print("markers present:", len(set(labels.tolist())), " nan labels:", bool(np.isnan(labels).any()))
print("all edges cross-side?", bool((side[edges[:, 0]] != side[edges[:, 1]]).all()))

## Step 1 — MCMC validation gate (§6)

Confirm the analytic mean/variance match explicit null simulations on a few (cell, motif) pairs **before** launching the full run. Two nulls: soft Chung–Lu (checks the formulas) and degree-preserving edge-swap (checks adequacy on the real graph). Do not submit jobs until the mean criterion (±3%) passes.

In [ ]:
# Pick 3 S001 cells spanning size (small / medium / large by edge count E).
def load_one(comp):
    el = pg.filter(components=[comp]).edgelist().to_df()
    return ct.load_cell(el[["umi1", "umi2", "marker_1", "marker_2"]], marker_to_idx)

import random
random.seed(0)
s001 = adata.obs_names[adata.obs["sample"] == "S001"].tolist()
probe = random.sample(s001, 24)
sizes = []
for c in probe:
    e, l, sd, _ = load_one(c)
    sizes.append((e.shape[0], c, e, l, sd))
sizes.sort(key=lambda x: x[0])
chosen = [sizes[1], sizes[len(sizes) // 2], sizes[-2]]   # small, median, large
for E, c, _, _, _ in chosen:
    print(f"{c[:8]}: E={E}")

In [ ]:
# Bipartite null validation. Triplet analytic uses the FIXED-DEGREE (configuration) null,
# so the bipartite edge-swap MC is the matching gate; chunglu (soft) is shown as reference
# and is expected to sit ~1/dbar ABOVE the analytic for wedges. Each null graph is drawn
# ONCE per sample (O(m) samplers), all motifs read off the same build_cell.
NS = 400
POP = 20.0   # "populated" motif threshold on analytic E (gate uses these; rare = tail-demo)

def _stats_from_graph(edges, labels, pairs, trips):
    A, L, d, twoE, N = ct.build_cell(edges, labels, K)
    pc = {}
    for (a, b) in pairs:
        pc[(a, b)] = (0.5 * float(N[labels == a][:, a].sum()) if a == b
                      else float(N[labels == a][:, b].sum()))
    wc = {}
    for (a, b, c) in trips:
        ia, ib, ic = labels == a, labels == b, labels == c
        wc[(a, b, c)] = float(N[ia][:, b] @ N[ia][:, c]
                              + N[ib][:, a] @ N[ib][:, c]
                              + N[ic][:, a] @ N[ic][:, b])
    return pc, wc

def analytic_and_mc(edges, labels, side, name):
    A, L, d, twoE, N = ct.build_cell(edges, labels, K)
    s1, r1, t1, s2, r2, t2 = ct.side_strengths(d, labels, side, K)
    m = d.sum() / 2.0
    EW = ct.triplet_mean(s1, r1, s2, r2, m)
    Vw = ct.triplet_var_lead(EW, s1, r1, t1, s2, r2, t2, m)
    mean_pair = (np.outer(s1, s2) + np.outer(s2, s1)) / m
    var_pair = mean_pair - (np.outer(r1, r2) + np.outer(r2, r1)) / m ** 2
    np.fill_diagonal(mean_pair, (s1 * s2) / m)
    np.fill_diagonal(var_pair, (s1 * s2) / m - (r1 * r2) / m ** 2)

    # ---- motif selection: populated by analytic value + a few rare (tail-demo) ----
    stot = s1 + s2
    present = np.array(sorted(set(labels.tolist())))
    cand = present[np.argsort(-stot[present])][:14].tolist()        # high-strength markers
    low = present[np.argsort(stot[present])][:6].tolist()           # low-strength markers
    pc = sorted(combinations(cand, 2), key=lambda p: -mean_pair[p])
    pop_pairs = [p for p in pc if mean_pair[p] >= POP][:4]
    pairs = pop_pairs + [(cand[0], cand[0])] + list(combinations(low, 2))[:1]
    tc = sorted(combinations(cand, 3), key=lambda t: -EW[t])
    pop_trips = [t for t in tc if EW[t] >= POP][:4]
    trips = pop_trips + list(combinations(low, 3))[:2]

    obs_p, obs_t = _stats_from_graph(edges, labels, pairs, trips)
    motifs = pairs + trips
    draws = {nl: {mo: np.empty(NS) for mo in motifs} for nl in ["chunglu", "edgeswap"]}
    for nl in ["chunglu", "edgeswap"]:
        for jx in range(NS):
            e = (ct.bipartite_chunglu_sample_edges(d, side, jx) if nl == "chunglu"
                 else ct.bipartite_edgeswap_sample(edges, side, nswap_mult=20, seed=jx))
            p_c, w_c = _stats_from_graph(e, labels, pairs, trips)
            for p in pairs:
                draws[nl][p][jx] = p_c[p]
            for tr in trips:
                draws[nl][tr][jx] = w_c[tr]

    rows, store = [], {}
    for nl in ["chunglu", "edgeswap"]:
        for p in pairs:
            arr = draws[nl][p]; anE = mean_pair[p]; anSD = np.sqrt(max(var_pair[p], 0))
            lbl = "·".join(marker_names[i] for i in p) + ("  (self)" if p[0] == p[1] else "")
            rows.append(dict(cell=name, null=nl, motif="pair", m=lbl, populated=anE >= POP,
                             obs=obs_p[p], an_E=anE, an_SD=anSD,
                             mc_mean=arr.mean(), mc_SD=arr.std()))
            store[(nl, "pair", p)] = (arr, obs_p[p], anE, anSD)
        for tr in trips:
            arr = draws[nl][tr]; anE = EW[tr]; anSD = np.sqrt(max(Vw[tr], 0))
            rows.append(dict(cell=name, null=nl, motif="triplet",
                             m="·".join(marker_names[i] for i in tr), populated=anE >= POP,
                             obs=obs_t[tr], an_E=anE, an_SD=anSD,
                             mc_mean=arr.mean(), mc_SD=arr.std()))
            store[(nl, "triplet", tr)] = (arr, obs_t[tr], anE, anSD)
    return pd.DataFrame(rows), store

tables, stores = [], {}
for E, c, e, l, sd in chosen:
    df, store = analytic_and_mc(e, l, sd, c[:8])
    tables.append(df); stores[c] = store
    print(f"done {c[:8]} (E={E})")
val = pd.concat(tables, ignore_index=True)
val["E_ratio"] = val["mc_mean"] / val["an_E"]
val["SD_ratio"] = val["mc_SD"] / val["an_SD"]
display(val.round(3))

In [ ]:
# Acceptance, on POPULATED motifs only (rare motifs have ~0 expectation -> meaningless ratios).
#  - edgeswap (bipartite, fixed degrees): the FAITHFUL null the triplet analytic now targets.
#    HARD GATE: edge-swap mean within +-3% (pooled Sigma mc / Sigma analytic, and median).
#  - chunglu (bipartite soft): reference only -- for wedges it sits ~1/dbar ABOVE the analytic
#    (soft vs fixed-degree); doublets match either null.
pop = val[val["populated"]].copy()

def pooled(sub):
    return sub["mc_mean"].sum() / sub["an_E"].sum()

print("POPULATED-motif ratios (MC / analytic):")
summ = (pop.groupby(["null", "motif"])
           .apply(lambda s: pd.Series({"n": len(s), "pooled_E": pooled(s),
                                        "median_E": s["E_ratio"].median(),
                                        "median_SD": s["SD_ratio"].median()}))
           .round(3))
print(summ)

es = pop[pop["null"] == "edgeswap"]
gate_ratio = pooled(es)
gate_ok = abs(gate_ratio - 1) < 0.03
print(f"\n[GATE] edge-swap pooled mean ratio (populated) = {gate_ratio:.3f}  "
      f"-> {'PASS' if gate_ok else 'FAIL'} (+-3%)")
print("Note: chunglu wedge ratio >1 is expected (soft null); edge-swap SD>MC is the "
      "conservative leading-order variance (Z slightly deflated, safe).")
print(f"Excluded {int((~val['populated']).sum())} rare (tail-demo) rows from the gate.")

# Null histograms: edge-swap (the gated null) on a few populated triplets, observed marked.
cmid = list(stores)[1]
trips = [k for k in stores[cmid]
         if k[0] == "edgeswap" and k[1] == "triplet" and stores[cmid][k][2] >= 20][:3]
if trips:
    fig, axs = plt.subplots(1, len(trips), figsize=(4 * len(trips), 3))
    for ax, k in zip(np.atleast_1d(axs), trips):
        mc, obs, E, SD = stores[cmid][k]
        ax.hist(mc, bins=30, color="#4c72b0", alpha=.7)
        ax.axvline(obs, color="r", label="observed")
        ax.axvline(E, color="k", ls="--", label="analytic E")
        ax.set_title("·".join(marker_names[i] for i in k[2]), fontsize=8)
    np.atleast_1d(axs)[0].legend(fontsize=7)
    plt.tight_layout()
print("\nGate passed -> proceed to Step 2." if gate_ok
      else "\nGate FAILED: debug strengths/side/m or the falling-factorial correction.")

## Step 2 — Submit per-sample jobs (LSF array, CPU)

In [ ]:
# Gate passed -> submit the full 15-sample LSF array (CPU `long` queue). Heavy compute on LSF.
# Single-sample smoke test instead:
#   !cd {BASE} && bsub -J "cl-S001" -env "SAMPLE=S001" < run_chunglu_triplets.lsf
(RESULTS / "logs").mkdir(parents=True, exist_ok=True)
!cd {BASE} && bsub < run_chunglu_triplets.lsf

# Monitor:
#   !bjobs -w | grep chunglu
#   !tail -n 20 {RESULTS}/logs/chunglu_*.out

In [ ]:
# Monitor jobs + completed outputs (re-run to refresh).
!bjobs -w | grep -i chunglu || echo "no chunglu jobs in queue"
done = sorted(p.name for p in OUT.glob("*_doublets.parquet"))
print(f"\n{len(done)}/15 samples done:", [n.split('_')[0] for n in done])
!tail -n 5 {RESULTS}/logs/chunglu_*.err 2>/dev/null | tail -n 30

## Step 3 — Aggregate across samples (run after jobs finish)

Pool the per-sample outputs and contrast conditions, blocked on `cell_system`, FDR via Benjamini–Yekutieli. Cells are the unit of replication.

In [ ]:
# Load per-sample outputs + attach metadata (run after the LSF jobs finish).
sample_meta = {
    "S001": {"time": "6h",  "condition": "Mock",         "target": "healthy B", "tcells": "healthy T"},
    "S002": {"time": "6h",  "condition": "Blinatumomab", "target": "healthy B", "tcells": "healthy T"},
    "S003": {"time": "48h", "condition": "Mock",         "target": "healthy B", "tcells": "healthy T"},
    "S004": {"time": "48h", "condition": "Blinatumomab", "target": "healthy B", "tcells": "healthy T"},
    "S005": {"time": "6h",  "condition": "Mock",         "target": "NALM-6",    "tcells": "healthy T"},
    "S006": {"time": "6h",  "condition": "Blinatumomab", "target": "NALM-6",    "tcells": "healthy T"},
    "S007": {"time": "48h", "condition": "Mock",         "target": "NALM-6",    "tcells": "healthy T"},
    "S008": {"time": "48h", "condition": "Blinatumomab", "target": "NALM-6",    "tcells": "healthy T"},
    "S009": {"time": "6h",  "condition": "Mock",         "target": "patient B", "tcells": "patient T"},
    "S010": {"time": "6h",  "condition": "Blinatumomab", "target": "patient B", "tcells": "patient T"},
    "S011": {"time": "48h", "condition": "Mock",         "target": "patient B", "tcells": "patient T"},
    "S012": {"time": "48h", "condition": "Blinatumomab", "target": "patient B", "tcells": "patient T"},
    "S013": {"time": "6h",  "condition": "Mock",         "target": "NALM-6",    "tcells": "patient T"},
    "S014": {"time": "6h",  "condition": "Blinatumomab", "target": "NALM-6",    "tcells": "patient T"},
    "S016": {"time": "48h", "condition": "Blinatumomab", "target": "NALM-6",    "tcells": "patient T"},
}
meta = pd.DataFrame(sample_meta).T

dbl = pd.concat([pd.read_parquet(p) for p in sorted(OUT.glob("*_doublets.parquet"))], ignore_index=True)
trp = pd.concat([pd.read_parquet(p) for p in sorted(OUT.glob("*_triplets_pooled.parquet"))], ignore_index=True)
for df in (dbl, trp):
    for c in ["condition", "time", "target", "tcells"]:
        df[c] = df["sample"].map(meta[c])
    df["cell_system"] = df["target"] + " + " + df["tcells"]
trp["mean_Z"] = trp["sum_Z"] / trp["n_cells"]
print(f"doublet rows: {len(dbl):,}   triplet pooled rows: {len(trp):,}")
print("\nTop triplets by pooled mean Z (co-proximity):")
display(trp.sort_values("mean_Z", ascending=False)
           .loc[:, ["sample", "marker_1", "marker_2", "marker_3", "mean_Z", "n_cells"]].head(10))

In [ ]:
trp.head()

In [ ]:
# Triplet differential: Blinatumomab vs Mock, blocked on cell_system, BY FDR.
# Only pooled per-cell Z moments were streamed (sum_Z, sum_Z2, n_cells), so cells are the
# unit and we use a Welch t-test reconstructed from those moments.
from scipy.stats import t as tdist
from statsmodels.stats.multitest import multipletests

key = ["marker_1", "marker_2", "marker_3"]
agg = (trp.groupby(["cell_system", "condition"] + key, observed=True)
          .agg(n=("n_cells", "sum"), sZ=("sum_Z", "sum"), sZ2=("sum_Z2", "sum"))
          .reset_index())
agg = agg[agg["n"] >= 30].copy()
agg["mean"] = agg["sZ"] / agg["n"]
agg["var"] = (agg["sZ2"] / agg["n"] - agg["mean"] ** 2).clip(lower=0)

b = agg[agg["condition"] == "Blinatumomab"].set_index(["cell_system"] + key)
m = agg[agg["condition"] == "Mock"].set_index(["cell_system"] + key)
j = b.join(m, lsuffix="_b", rsuffix="_m", how="inner")

# Welch t-test from group moments
se = np.sqrt(j["var_b"] / j["n_b"] + j["var_m"] / j["n_m"])
se = se.replace(0, np.nan)
tstat = (j["mean_b"] - j["mean_m"]) / se
dfree = se ** 4 / ((j["var_b"] / j["n_b"]) ** 2 / (j["n_b"] - 1)
                   + (j["var_m"] / j["n_m"]) ** 2 / (j["n_m"] - 1))
res_t = j.reset_index()[["cell_system"] + key].copy()
res_t["dZ"] = (j["mean_b"] - j["mean_m"]).values
res_t["p"] = 2 * tdist.sf(np.abs(tstat), dfree) if len(j) else []
res_t["n_blin"] = j["n_b"].values
res_t["n_mock"] = j["n_m"].values
res_t = res_t.dropna(subset=["p"])
if len(res_t):
    res_t["fdr"] = multipletests(res_t["p"], method="fdr_by")[1]
    res_t = res_t.sort_values("p")
    print(f"{(res_t.fdr < 0.05).sum()} triplets FDR<0.05 of {len(res_t)} tested")
    display(res_t.head(15))
else:
    print("No triplets with both arms present at n>=30.")

In [ ]:
# Doublet differential: Blinatumomab vs Mock, per cell, blocked on cell_system, BY FDR.
# Cells are the unit (per-cell join_count_z stored), so use Mann-Whitney directly.
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

dbl["pair"] = dbl["marker_1"] + "/" + dbl["marker_2"]
MIN_CELLS = 30
res_d = []
for (sysname, pair), sub in dbl.groupby(["cell_system", "pair"], observed=True):
    a = sub.loc[sub["condition"] == "Blinatumomab", "join_count_z"]
    b = sub.loc[sub["condition"] == "Mock", "join_count_z"]
    if len(a) < MIN_CELLS or len(b) < MIN_CELLS:
        continue
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    res_d.append(dict(cell_system=sysname, pair=pair, dZ=a.mean() - b.mean(),
                      p=p, n_blin=len(a), n_mock=len(b)))
res_d = pd.DataFrame(res_d)
if len(res_d):
    res_d["fdr"] = multipletests(res_d["p"], method="fdr_by")[1]
    res_d = res_d.sort_values("p")
    print(f"{(res_d.fdr < 0.05).sum()} pairs FDR<0.05")
    display(res_d.head(15))

    # volcano (one system)
    sysname = res_d["cell_system"].value_counts().idxmax()
    v = res_d[res_d["cell_system"] == sysname]
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.scatter(v["dZ"], -np.log10(v["p"]), s=6,
               c=np.where(v["fdr"] < 0.05, "#c44e52", "#bbbbbb"))
    ax.axhline(-np.log10(0.05), ls="--", lw=.7, color="k")
    ax.set_xlabel("ΔZ  (Blin − Mock)"); ax.set_ylabel("-log10 p")
    ax.set_title(f"Doublet colocalization shift — {sysname}")
    plt.tight_layout()
else:
    print("Not enough per-arm cells; lower MIN_CELLS or pool systems.")

## Methods note

- **Graph:** unweighted simple molecule-level PNA graph per cell — node = one UMI (one marker), edge = proximity ligation. The graph is **bipartite**: `umi1`/`umi2` are disjoint node sets (Step 0 prints overlap = 0) and edges form only across sides. Edges de-duplicated, no self-loops; `read_count`/`uei_count` ignored (degree-only).
- **Strengths** split by side, normaliser **`m` = number of edges** (`2E = 2m`): `s1,r1,t1` (side 1), `s2,r2,t2` (side 2).
- **Doublets** `J_AB` (soft = fixed-degree for pairs; validated, edge-swap ratio ≈ 0.997): `E = (s1_A s2_B + s2_A s1_B)/m`, `Var = E − (r1_A r2_B + r2_A r1_B)/m²`. Self-pairs (A=A polarization) use `E = s1_A s2_A/m` (single configuration) + own variance, observed `J[A,A]/2`.
- **Triplets** `W_{ABC}` (open wedge) use the **fixed-degree (configuration) null**, matching the real graph (the soft Chung–Lu form overestimates by `~1−1/d̄`; at `d̄≈3.3` that is ~20–25%). Falling-factorial rule: a node anchoring `k` motif edges contributes `d^{(k)}` not `d^k` — wedge **center** (anchors 2) uses `r−s = Σd(d−1)`; spokes (anchor 1) use `s`; 3-edge self-moments use `t−3r+2s`. Mean matches the edge-swap MC to **±0.2%**. The leading-order variance with the same substitution is **conservative** (analytic SD ≈ 1.3–1.4× the edge-swap SD → Z slightly deflated, safe); reported, not gated. No triangles in a bipartite graph. Stream-aggregated per sample (`ΣW, ΣEW, ΣZ, ΣZ², n_cells` over all C(K,3)).
- **Observed counts unchanged** (`N=A@L`, `J=LᵀN`, `W=Σ Nᵍᵀ Nᵍ`); only the null strengths/formulas changed.
- **QC:** cells from `cache/adata_annotated.h5ad` (`tau_type==normal` & `n_umi≥25000`). K = 159 markers (global, `adata.var_names`).
- **Validation:** the bipartite **edge-swap** MC (degree- and bipartite-preserving) is the gate — analytic mean within ±3% on **populated** motifs (`E≥20`); rare motifs excluded (meaningless ratios). The soft **chunglu** MC is shown for reference and sits `~1/d̄` above the analytic for wedges. O(m) samplers (Norros–Reittu Poisson; configuration-model permutation).
- **Aggregation:** per-cell colocalization; condition contrasts blocked on `cell_system`; FDR via Benjamini–Yekutieli. Cells are the unit; small-`n` arms under-powered.